In [2]:
import os
import time
import platform
from functools import wraps
from datetime import datetime

import numpy as np
from numpy.typing import NDArray

In [3]:
def timestamped_print(string : str) -> None:
    """ Function to print messages with a timestamp """
    cur_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    output = f"{cur_time} {string}"
    print(output)



def benchmark(num_runs: int = 10, num_repetitions: int = 100):
    if num_runs <= 0:
                raise ValueError("num_runs must be a positive integer greater than 1")
    
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            durations = []

            for _ in range(num_runs):
                start_time = time.perf_counter()

                for _ in range(num_repetitions):
                    result = func(*args, **kwargs)
            
                end_time = time.perf_counter()
                duration = end_time - start_time
                durations.append(duration)

            mean = np.mean(durations)
            std = np.std(durations)

            run_str = "run" if num_runs == 1 else "runs"
       
            message = f"{func.__name__} benchmark: {mean:.5f} ± {std:.5f} seconds per loop (mean ± std. dev. of {num_runs} {run_str}, {num_repetitions} repetitions each"
            timestamped_print(message)

            return result
        return wrapper
    return decorator

In [4]:
@benchmark(1, 5)
def test_transpose(a: NDArray):
    result = np.transpose(a)
    return result

In [5]:
DIM = 2500
NUM_RUNS = 10
NUM_REPETITIONS = 1
SEED = 69
SKIP_BENCHMARKS = False


if not SKIP_BENCHMARKS:
    @benchmark(NUM_RUNS, NUM_REPETITIONS)
    def test_transpose(a: NDArray):
        result = np.transpose(a)
        return result

    @benchmark(NUM_RUNS, NUM_REPETITIONS)
    def test_sum(a: NDArray):
        result = np.sum(a)
        return result

    @benchmark(NUM_RUNS, NUM_REPETITIONS)
    def test_dot(a: NDArray, b: NDArray):
        result = np.dot(a, b)
        return result

    @benchmark(NUM_RUNS, NUM_REPETITIONS)
    def test_eigvals(a: NDArray):
        result = np.linalg.eig(a)
        return result

    @benchmark(NUM_RUNS, NUM_REPETITIONS)
    def test_fft(a: NDArray):
        result = np.fft.fft(a)
        return result

    @benchmark(NUM_RUNS, NUM_REPETITIONS)
    def test_svd(mat: NDArray):
        result = np.linalg.svd(mat, full_matrices = False)
        return result

    rng = np.random.default_rng(SEED)
    size = (DIM, DIM)

    a = rng.random(size=size)
    b = rng.random(size=size)

    test_transpose(a)
    test_sum(a)
    test_dot(a, b)
    test_eigvals(a)
    test_fft(a)
    test_svd(a)

2024-03-21 11:59:09 test_transpose benchmark: 0.00000 ± 0.00001 seconds per loop (mean ± std. dev. of 10 runs, 1 repetitions each
2024-03-21 11:59:09 test_sum benchmark: 0.00100 ± 0.00052 seconds per loop (mean ± std. dev. of 10 runs, 1 repetitions each
2024-03-21 11:59:10 test_dot benchmark: 0.08954 ± 0.00240 seconds per loop (mean ± std. dev. of 10 runs, 1 repetitions each
2024-03-21 11:59:50 test_eigvals benchmark: 4.01435 ± 0.20051 seconds per loop (mean ± std. dev. of 10 runs, 1 repetitions each
2024-03-21 11:59:50 test_fft benchmark: 0.03367 ± 0.00185 seconds per loop (mean ± std. dev. of 10 runs, 1 repetitions each
2024-03-21 12:00:11 test_svd benchmark: 2.10382 ± 0.08573 seconds per loop (mean ± std. dev. of 10 runs, 1 repetitions each


In [6]:
# First check if Python is being run natively on the MX processor
# or if it is being run in a virtual environment. Specifically, if 
# your computer is running Rosetta Python, this should print out 'i386'
# instead of 'arm' as the processor type.
processor_type = platform.processor()
timestamped_print(f"Processor type: {processor_type}")

# Next print the virtual environment that Python is running on and that is being benchmarked.
virtual_env = os.getenv("VIRTUAL_ENV")
timestamped_print(f"Benchmarking for virtual environment: {virtual_env}")

# Print the numpy configuration
np_config = np.show_config(mode="dicts")
build_dependencies = np_config["Build Dependencies"]

# Then print out the build dependencies for numpy/blas
for framework, info in build_dependencies.items():
    timestamped_print(f"{framework} build dependencies:")
    for key, value in info.items():
        print(f"\t{key}: {value}")

2024-03-21 12:00:11 Processor type: arm
2024-03-21 12:00:11 Benchmarking for virtual environment: /Users/varb2323/Projects/dynamiq-sim/.venv
2024-03-21 12:00:11 blas build dependencies:
	name: accelerate
	found: True
	version: unknown
	detection method: system
	include directory: unknown
	lib directory: unknown
	openblas configuration: unknown
	pc file directory: unknown
2024-03-21 12:00:11 lapack build dependencies:
	name: dep4343479136
	found: True
	version: 1.26.4
	detection method: internal
	include directory: unknown
	lib directory: unknown
	openblas configuration: unknown
	pc file directory: unknown
